In [37]:
import os
import glob
import shutil
from Bio.PDB import MMCIFParser

# ——— CONFIGURATION ———
CIF_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/Train_dataset'
MSA_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA'
NEW_MSA_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/CORRECTED_MSA'

# Create new MSA directory if it doesn't exist
os.makedirs(NEW_MSA_DIR, exist_ok=True)

parser = MMCIFParser(QUIET=True)

def extract_rna_sequence(cif_path: str) -> str:
    """Extract only A/C/G/U from the first model of the CIF."""
    structure = parser.get_structure('', cif_path)
    seq = []
    for chain in structure[0]:
        for residue in chain:
            res = residue.get_resname().strip()
            if res in ('A','C','G','U'):
                seq.append(res)
    return ''.join(seq)

def read_first_msa_sequence(msa_path: str) -> str:
    """Read the very first FASTA entry (after '>query')."""
    seq_lines = []
    with open(msa_path) as f:
        # skip header
        for line in f:
            if line.startswith('>'):
                break
        # read until next header or EOF
        for line in f:
            if line.startswith('>'):
                break
            seq_lines.append(line.strip())
    return ''.join(seq_lines)

def create_new_msa_file(output_path: str, sequence: str, tid: str):
    """Create a new MSA file with just the query sequence."""
    with open(output_path, 'w') as f:
        f.write(f">query_{tid}\n")
        # Write sequence in lines of 80 characters (standard FASTA format)
        for i in range(0, len(sequence), 80):
            f.write(sequence[i:i+80] + '\n')

total = 0
copied_count = 0
created_count = 0
no_msa_count = 0

print("Processing MSA files...")
print("=" * 50)

for cif_file in glob.glob(os.path.join(CIF_DIR, '*.cif')):
    tid = os.path.splitext(os.path.basename(cif_file))[0]
    msa_file = os.path.join(MSA_DIR, f"{tid}.MSA.fasta")
    new_msa_file = os.path.join(NEW_MSA_DIR, f"{tid}.MSA.fasta")
    
    if not os.path.exists(msa_file):
        # No MSA file exists, create one with CIF sequence
        cif_seq = extract_rna_sequence(cif_file)
        if cif_seq:  # Only create if we have a valid sequence
            create_new_msa_file(new_msa_file, cif_seq, tid)
            created_count += 1
            print(f"Created new MSA for {tid} (no original MSA)")
        no_msa_count += 1
        continue

    cif_seq = extract_rna_sequence(cif_file)
    msa_seq = read_first_msa_sequence(msa_file)
    total += 1

    # Check if CIF sequence is contained in MSA sequence
    if cif_seq in msa_seq:
        # MSA is correct, copy the original file
        shutil.copy2(msa_file, new_msa_file)
        copied_count += 1
        print(f"✓ Copied original MSA for {tid}")
    else:
        # Mismatch, create new MSA with CIF sequence
        create_new_msa_file(new_msa_file, cif_seq, tid)
        created_count += 1
        print(f"✗ Created corrected MSA for {tid} (sequence mismatch)")

# Final reporting
print("\n" + "=" * 50)
print("SUMMARY:")
print(f"Total CIF files processed: {total + no_msa_count}")
print(f"  Files with existing MSA: {total}")
print(f"  Files without MSA: {no_msa_count}")
print(f"\nMSA Processing Results:")
print(f"  ✓ Original MSA files copied (sequence match): {copied_count}")
print(f"  ✗ New MSA files created (mismatch/missing): {created_count}")
print(f"\nNew MSA directory created at: {NEW_MSA_DIR}")
print(f"Total files in new directory: {copied_count + created_count}")

Processing MSA files...
✗ Created corrected MSA for 7N2U_Dt (sequence mismatch)
✓ Copied original MSA for 8EMM_Y
✓ Copied original MSA for 6XMG_C
✓ Copied original MSA for 7WB1_D
✓ Copied original MSA for 8VPV_A
✓ Copied original MSA for 7PKT_6
✓ Copied original MSA for 7VTN_B
✓ Copied original MSA for 7PKQ_4
✗ Created corrected MSA for 7XUE_R (sequence mismatch)
✗ Created corrected MSA for 8CH6_d (sequence mismatch)
✗ Created corrected MSA for 7Y7C_V (sequence mismatch)
✓ Copied original MSA for 8DVR_B
✓ Copied original MSA for 7EAG_C
✓ Copied original MSA for 9G7C_A
✗ Created corrected MSA for 7XHT_B (sequence mismatch)
✓ Copied original MSA for 7UIN_B
✓ Copied original MSA for 7R6N_A
✓ Copied original MSA for 8PM0_V
✗ Created corrected MSA for 7O80_AT (sequence mismatch)
✓ Copied original MSA for 8Q87_B7
✓ Copied original MSA for 7U2A_C
✓ Copied original MSA for 8APO_A9
✓ Copied original MSA for 8ETG_6
✗ Created corrected MSA for 7OPE_2 (sequence mismatch)
✓ Copied original MSA for 

In [38]:
import os
import glob
import shutil
from Bio.PDB import MMCIFParser

# ——— CONFIGURATION ———
CIF_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/Test_dataset'
MSA_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA'
NEW_MSA_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/CORRECTED_MSA'

# Create new MSA directory if it doesn't exist
os.makedirs(NEW_MSA_DIR, exist_ok=True)

parser = MMCIFParser(QUIET=True)

def extract_rna_sequence(cif_path: str) -> str:
    """Extract only A/C/G/U from the first model of the CIF."""
    structure = parser.get_structure('', cif_path)
    seq = []
    for chain in structure[0]:
        for residue in chain:
            res = residue.get_resname().strip()
            if res in ('A','C','G','U'):
                seq.append(res)
    return ''.join(seq)

def read_first_msa_sequence(msa_path: str) -> str:
    """Read the very first FASTA entry (after '>query')."""
    seq_lines = []
    with open(msa_path) as f:
        # skip header
        for line in f:
            if line.startswith('>'):
                break
        # read until next header or EOF
        for line in f:
            if line.startswith('>'):
                break
            seq_lines.append(line.strip())
    return ''.join(seq_lines)

def create_new_msa_file(output_path: str, sequence: str, tid: str):
    """Create a new MSA file with just the query sequence."""
    with open(output_path, 'w') as f:
        f.write(f">query_{tid}\n")
        # Write sequence in lines of 80 characters (standard FASTA format)
        for i in range(0, len(sequence), 80):
            f.write(sequence[i:i+80] + '\n')

total = 0
copied_count = 0
created_count = 0
no_msa_count = 0

print("Processing MSA files...")
print("=" * 50)

for cif_file in glob.glob(os.path.join(CIF_DIR, '*.cif')):
    tid = os.path.splitext(os.path.basename(cif_file))[0]
    msa_file = os.path.join(MSA_DIR, f"{tid}.MSA.fasta")
    new_msa_file = os.path.join(NEW_MSA_DIR, f"{tid}.MSA.fasta")
    
    if not os.path.exists(msa_file):
        # No MSA file exists, create one with CIF sequence
        cif_seq = extract_rna_sequence(cif_file)
        if cif_seq:  # Only create if we have a valid sequence
            create_new_msa_file(new_msa_file, cif_seq, tid)
            created_count += 1
            print(f"Created new MSA for {tid} (no original MSA)")
        no_msa_count += 1
        continue

    cif_seq = extract_rna_sequence(cif_file)
    msa_seq = read_first_msa_sequence(msa_file)
    total += 1

    # Check if CIF sequence is contained in MSA sequence
    if cif_seq in msa_seq:
        # MSA is correct, copy the original file
        shutil.copy2(msa_file, new_msa_file)
        copied_count += 1
        print(f"✓ Copied original MSA for {tid}")
    else:
        # Mismatch, create new MSA with CIF sequence
        create_new_msa_file(new_msa_file, cif_seq, tid)
        created_count += 1
        print(f"✗ Created corrected MSA for {tid} (sequence mismatch)")

# Final reporting
print("\n" + "=" * 50)
print("SUMMARY:")
print(f"Total CIF files processed: {total + no_msa_count}")
print(f"  Files with existing MSA: {total}")
print(f"  Files without MSA: {no_msa_count}")
print(f"\nMSA Processing Results:")
print(f"  ✓ Original MSA files copied (sequence match): {copied_count}")
print(f"  ✗ New MSA files created (mismatch/missing): {created_count}")
print(f"\nNew MSA directory created at: {NEW_MSA_DIR}")
print(f"Total files in new directory: {copied_count + created_count}")

Processing MSA files...
✓ Copied original MSA for 9LCR_B
✗ Created corrected MSA for 9EY1_T (sequence mismatch)
✗ Created corrected MSA for 9J6Y_E (sequence mismatch)
✗ Created corrected MSA for 9LMF_F (sequence mismatch)
✗ Created corrected MSA for 8X9M_A (sequence mismatch)
✓ Copied original MSA for 9DTR_2
✓ Copied original MSA for 9IS7_B
✗ Created corrected MSA for 9L5R_2 (sequence mismatch)
✗ Created corrected MSA for 9J3T_B (sequence mismatch)
✓ Copied original MSA for 8X9O_A
✓ Copied original MSA for 9FIA_bT
✓ Copied original MSA for 9FI8_hB
✓ Copied original MSA for 9G6K_lG
✗ Created corrected MSA for 9DTR_6 (sequence mismatch)
✓ Copied original MSA for 8SYK_C
✓ Copied original MSA for 9L5R_6
✓ Copied original MSA for 9FCV_B
✓ Copied original MSA for 8YII_C
✗ Created corrected MSA for 9AR6_B (sequence mismatch)
✓ Copied original MSA for 8XCC_B
✓ Copied original MSA for 8XTP_B
✗ Created corrected MSA for 9BLM_A (sequence mismatch)
✓ Copied original MSA for 9ESI_5
✓ Copied origina

In [35]:
import os
import glob
from Bio.PDB import MMCIFParser

# 1) Configuration — update these paths as needed
CIF_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/Train_dataset'
MSA_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA'

parser = MMCIFParser(QUIET=True)

def extract_rna_sequence(cif_path: str) -> str:
    """Extract only A/C/G/U from the first model of the CIF."""
    structure = parser.get_structure('', cif_path)
    seq = []
    for chain in structure[0]:
        for residue in chain:
            res = residue.get_resname().strip()
            if res in ('A','C','G','U'):
                seq.append(res)
    return ''.join(seq)

def read_first_msa_sequence(msa_path: str) -> str:
    """Read lines after the first header until the next '>' or EOF."""
    seq = []
    with open(msa_path) as f:
        for line in f:
            if line.startswith('>'):
                # skip the header line
                header = line
                break
        for line in f:
            if line.startswith('>'):
                break
            seq.append(line.strip())
    return ''.join(seq)

# 2) Compare
total, match, mismatch = 0, 0, 0
mismatch_ids = []

for cif_file in glob.glob(os.path.join(CIF_DIR, '*.cif')):
    tid = os.path.splitext(os.path.basename(cif_file))[0]
    msa_path = os.path.join(MSA_DIR, f"{tid}.MSA.fasta")
    if not os.path.exists(msa_path):
        # skip if you never generated an MSA
        continue

    cif_seq = extract_rna_sequence(cif_file)
    msa_seq = read_first_msa_sequence(msa_path)
    total += 1

    if cif_seq == msa_seq:
        match += 1
    else:
        mismatch += 1
        mismatch_ids.append(tid)

# 3) Print high-level stats
print(f"Total compared: {total}")
print(f"  ✓ Matches:    {match}")
print(f"  ✗ Mismatches: {mismatch}")
print()
if mismatch_ids:
    print("Example mismatches (up to 10):")
    for tid in mismatch_ids[:10]:
        print(" ", tid)


Total compared: 588
  ✓ Matches:    6
  ✗ Mismatches: 582

Example mismatches (up to 10):
  7N2U_Dt
  8EMM_Y
  6XMG_C
  7WB1_D
  8VPV_A
  7PKT_6
  7VTN_B
  7PKQ_4
  7XUE_R
  8CH6_d


In [6]:
import os
import glob
from Bio.PDB import MMCIFParser
from difflib import SequenceMatcher

# ——— CONFIGURATION ———
CIF_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/Train_dataset'
MSA_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA'
SIMILARITY_THRESHOLD = 0.90  # you can still use this for high-level stats

parser = MMCIFParser(QUIET=True)

def extract_rna_sequence(cif_path: str) -> str:
    structure = parser.get_structure('', cif_path)
    seq = []
    for chain in structure[0]:
        for residue in chain:
            res = residue.get_resname().strip()
            if res in ('A','C','G','U'):
                seq.append(res)
    return ''.join(seq)

def read_first_msa_sequence(msa_path: str) -> str:
    seq = []
    with open(msa_path) as f:
        # skip header
        for line in f:
            if line.startswith('>'):
                break
        # read until next header or EOF
        for line in f:
            if line.startswith('>'):
                break
            seq.append(line.strip())
    return ''.join(seq)

# 1) Gather all mismatches with their identity ratios
mismatches = []
for cif_file in glob.glob(os.path.join(CIF_DIR, '*.cif')):
    tid = os.path.splitext(os.path.basename(cif_file))[0]
    msa_file = os.path.join(MSA_DIR, f"{tid}.MSA.fasta")
    if not os.path.exists(msa_file):
        continue

    cif_seq = extract_rna_sequence(cif_file)
    msa_seq = read_first_msa_sequence(msa_file)
    if cif_seq != msa_seq:
        ratio = SequenceMatcher(None, cif_seq, msa_seq).ratio()
        mismatches.append((tid, len(cif_seq), len(msa_seq), ratio))

total_mm = len(mismatches)
mostly_similar = [m for m in mismatches if m[3] >= SIMILARITY_THRESHOLD]

# 2) Print high-level summary
print(f"Total mismatches: {total_mm}")
print(f"  → ≥{int(SIMILARITY_THRESHOLD*100)}% identity: {len(mostly_similar)}")
print(f"  → <{int(SIMILARITY_THRESHOLD*100)}% identity: {total_mm - len(mostly_similar)}\n")

# 3) Extract the 10 most dissimilar (lowest identity) entries
dissimilar = sorted(mismatches, key=lambda x: x[3])[:10]

print("Top 10 most dissimilar mismatches (ID, len_CIF, len_MSA, identity):")
for tid, l1, l2, r in dissimilar:
    print(f"  • {tid}: {l1} vs {l2}  →  {r:.3f}")


Total mismatches: 208
  → ≥90% identity: 148
  → <90% identity: 60

Top 10 most dissimilar mismatches (ID, len_CIF, len_MSA, identity):
  • 7XHT_B: 179 vs 228  →  0.000
  • 7SYO_z: 166 vs 400  →  0.000
  • 7V9X_D: 0 vs 39  →  0.000
  • 8QBK_Q: 0 vs 85  →  0.000
  • 8QBL_L: 0 vs 37  →  0.000
  • 8CXF_A: 0 vs 26  →  0.000
  • 8RUH_A: 390 vs 391  →  0.000
  • 8UPO_5: 0 vs 26  →  0.000
  • 8RKT_1: 246 vs 261  →  0.000
  • 8A98_3: 160 vs 216  →  0.000


In [ ]:
import os
import glob
from Bio.PDB import MMCIFParser

# ——— CONFIGURATION ———
CIF_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/Train_dataset'
MSA_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/FINAL_MSA'

parser = MMCIFParser(QUIET=True)

def extract_rna_sequence(cif_path: str) -> str:
    """Extract only A/C/G/U from the first model of the CIF."""
    structure = parser.get_structure('', cif_path)
    seq = []
    for chain in structure[0]:
        for residue in chain:
            res = residue.get_resname().strip()
            if res in ('A','C','G','U'):
                seq.append(res)
    return ''.join(seq)

def read_first_msa_sequence(msa_path: str) -> str:
    """Read the very first FASTA entry (after '>query')."""
    seq_lines = []
    with open(msa_path) as f:
        # skip header
        for line in f:
            if line.startswith('>'):
                break
        # read until next header or EOF
        for line in f:
            if line.startswith('>'):
                break
            seq_lines.append(line.strip())
    return ''.join(seq_lines)

total = 0
subset_count = 0
no_subset_ids = []

for cif_file in glob.glob(os.path.join(CIF_DIR, '*.cif')):
    tid = os.path.splitext(os.path.basename(cif_file))[0]
    msa_file = os.path.join(MSA_DIR, f"{tid}.MSA.fasta")
    if not os.path.exists(msa_file):
        # skip entries with no MSA
        continue

    cif_seq = extract_rna_sequence(cif_file)
    msa_seq = read_first_msa_sequence(msa_file)
    total += 1

    # check substring
    if cif_seq in msa_seq:
        subset_count += 1
    else:
        no_subset_ids.append(tid)

# reporting
print(f"Total compared: {total}")
print(f"  ✓ MSA contains full CIF sequence:     {subset_count}")
print(f"  ✗ MSA does NOT contain full sequence:  {total - subset_count}")

if no_subset_ids:
    print("\nExamples of failures (up to 10):")
    for tid in no_subset_ids[:10]:
        print(" ", tid)


Total compared: 588
  ✓ MSA contains full CIF sequence:     467
  ✗ MSA does NOT contain full sequence:  121

Examples of failures (up to 10):
  7N2U_Dt
  7XUE_R
  8CH6_d
  7Y7C_V
  7XHT_B
  7O80_AT
  7OPE_2
  8HBA_B
  8HKU_AETN
  8J7R_C


In [ ]:
import os
import glob
from Bio.PDB import MMCIFParser

# ——— CONFIGURATION ———
CIF_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/Test_dataset'
MSA_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA'

parser = MMCIFParser(QUIET=True)

def extract_rna_sequence(cif_path: str) -> str:
    """Extract only A/C/G/U from the first model of the CIF."""
    structure = parser.get_structure('', cif_path)
    seq = []
    for chain in structure[0]:
        for residue in chain:
            res = residue.get_resname().strip()
            if res in ('A','C','G','U'):
                seq.append(res)
    return ''.join(seq)

def read_first_msa_sequence(msa_path: str) -> str:
    """Read the very first FASTA entry (after '>query')."""
    seq_lines = []
    with open(msa_path) as f:
        # skip header
        for line in f:
            if line.startswith('>'):
                break
        # read until next header or EOF
        for line in f:
            if line.startswith('>'):
                break
            seq_lines.append(line.strip())
    return ''.join(seq_lines)

total = 0
subset_count = 0
no_subset_ids = []

for cif_file in glob.glob(os.path.join(CIF_DIR, '*.cif')):
    tid = os.path.splitext(os.path.basename(cif_file))[0]
    msa_file = os.path.join(MSA_DIR, f"{tid}.MSA.fasta")
    if not os.path.exists(msa_file):
        # skip entries with no MSA
        continue

    cif_seq = extract_rna_sequence(cif_file)
    msa_seq = read_first_msa_sequence(msa_file)
    total += 1

    # check substring
    if cif_seq in msa_seq:
        subset_count += 1
    else:
        no_subset_ids.append(tid)

# reporting
print(f"Total compared: {total}")
print(f"  ✓ MSA contains full CIF sequence:     {subset_count}")
print(f"  ✗ MSA does NOT contain full sequence:  {total - subset_count}")

if no_subset_ids:
    print("\nExamples of failures (up to 10):")
    for tid in no_subset_ids[:10]:
        print(" ", tid)


Total compared: 27
  ✓ MSA contains full CIF sequence:     18
  ✗ MSA does NOT contain full sequence:  9

Examples of failures (up to 10):
  9EY1_T
  9J6Y_E
  9LMF_F
  8X9M_A
  9L5R_2
  9J3T_B
  9DTR_6
  9AR6_B
  9BLM_A


In [33]:
import os
import glob
from Bio.PDB import MMCIFParser

# ——— CONFIGURATION ———
CIF_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/Train_dataset'
MSA_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA'
CROPPED_MSA_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/CROPPED_MSA'

# Create output directory if it doesn't exist
os.makedirs(CROPPED_MSA_DIR, exist_ok=True)

parser = MMCIFParser(QUIET=True)

def extract_rna_sequence(cif_path: str) -> str:
    """Extract only A/C/G/U from the first model of the CIF."""
    structure = parser.get_structure('', cif_path)
    seq = []
    for chain in structure[0]:
        for residue in chain:
            res = residue.get_resname().strip()
            if res in ('A','C','G','U'):
                seq.append(res)
    return ''.join(seq)

def read_msa_file(msa_path: str) -> list:
    """Read the entire MSA file and return list of (header, sequence) tuples."""
    sequences = []
    current_header = None
    current_seq = []
    
    with open(msa_path) as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                # Save previous sequence if exists
                if current_header is not None:
                    sequences.append((current_header, ''.join(current_seq)))
                # Start new sequence
                current_header = line
                current_seq = []
            else:
                current_seq.append(line)
        
        # Save last sequence
        if current_header is not None:
            sequences.append((current_header, ''.join(current_seq)))
    
    return sequences

def find_cif_sequence_in_query(cif_seq: str, query_seq: str) -> tuple:
    """Find the start and end positions of CIF sequence in query sequence."""
    if cif_seq in query_seq:
        start_pos = query_seq.find(cif_seq)
        end_pos = start_pos + len(cif_seq)
        return start_pos, end_pos
    return None, None

def crop_msa_alignment(msa_sequences: list, start_pos: int, end_pos: int) -> list:
    """Crop all sequences in the MSA to the specified positions."""
    cropped_sequences = []
    for header, sequence in msa_sequences:
        cropped_seq = sequence[start_pos:end_pos]
        cropped_sequences.append((header, cropped_seq))
    return cropped_sequences

def write_cropped_msa(cropped_sequences: list, output_path: str):
    """Write the cropped MSA to a new file."""
    with open(output_path, 'w') as f:
        for header, sequence in cropped_sequences:
            f.write(f"{header}\n")
            f.write(f"{sequence}\n")

total = 0
subset_count = 0
cropped_count = 0
no_msa_count = 0
failed_crop_count = 0
no_subset_ids = []

print("Processing CIF files and checking MSA compatibility...")
print("=" * 60)

for cif_file in glob.glob(os.path.join(CIF_DIR, '*.cif')):
    tid = os.path.splitext(os.path.basename(cif_file))[0]
    msa_file = os.path.join(MSA_DIR, f"{tid}.MSA.fasta")
    
    if not os.path.exists(msa_file):
        # skip entries with no MSA
        no_msa_count += 1
        continue

    cif_seq = extract_rna_sequence(cif_file)
    msa_sequences = read_msa_file(msa_file)
    
    if not msa_sequences:
        print(f"  Warning: Empty MSA file for {tid}")
        continue
    
    # Get the query sequence (first sequence)
    query_header, query_seq = msa_sequences[0]
    total += 1

    # Check if CIF sequence is a substring of query sequence
    if cif_seq in query_seq:
        subset_count += 1
    else:
        no_subset_ids.append(tid)
        
        # Try to find the CIF sequence in the query and crop accordingly
        start_pos, end_pos = find_cif_sequence_in_query(cif_seq, query_seq)
        
        if start_pos is not None:
            # Crop all sequences in the MSA
            cropped_sequences = crop_msa_alignment(msa_sequences, start_pos, end_pos)
            
            # Write cropped MSA
            output_path = os.path.join(CROPPED_MSA_DIR, f"{tid}.MSA.fasta")
            write_cropped_msa(cropped_sequences, output_path)
            cropped_count += 1
            print(f"  ✓ Cropped MSA for {tid}: positions {start_pos}-{end_pos}")
        else:
            # CIF sequence not found as substring - this shouldn't happen with your logic
            # but let's handle it gracefully
            print(f"  ✗ Could not find CIF sequence in query for {tid}")
            print(f"    CIF seq: {cif_seq[:50]}{'...' if len(cif_seq) > 50 else ''}")
            print(f"    Query:   {query_seq[:50]}{'...' if len(query_seq) > 50 else ''}")
            failed_crop_count += 1

# Final reporting
print("\n" + "=" * 60)
print("SUMMARY REPORT:")
print("=" * 60)
print(f"Total CIF files found:                  {total + no_msa_count}")
print(f"  - Files with MSA available:           {total}")
print(f"  - Files without MSA:                  {no_msa_count}")
print(f"\nMSA Compatibility Results:")
print(f"  ✓ CIF sequence found in MSA query:    {subset_count}")
print(f"  📝 Required cropping:                  {len(no_subset_ids)}")
print(f"    - Successfully cropped:             {cropped_count}")
print(f"    - Failed to crop:                   {failed_crop_count}")
print(f"\nOutput directory: {CROPPED_MSA_DIR}")

if cropped_count > 0:
    print(f"\nSuccessfully created {cropped_count} cropped MSA files")

if no_subset_ids:
    print(f"\nFiles that required cropping (first 10 of {len(no_subset_ids)}):")
    for tid in no_subset_ids[:10]:
        print(f"  - {tid}")
    
    if len(no_subset_ids) > 10:
        print(f"  ... and {len(no_subset_ids) - 10} more")

print("\n" + "=" * 60)
print("Process completed!")

Processing CIF files and checking MSA compatibility...
  ✗ Could not find CIF sequence in query for 7N2U_Dt
    CIF seq: GCCCGGAAGCUCAGUUGGGAGAGCAGGGGAUGAAACCCCGUCCUUGGUCG...
    Query:   GCCCGGAUAGCUCAGUUGGGAGAGCAGGGGAUUGAAAAUCCCCGUGXCCU...
  ✗ Could not find CIF sequence in query for 7XUE_R
    CIF seq: UAGACGAACGGCGCGUCUUUAAACCAUGCGUCGGGAGCGCGGCGGGUUCA...
    Query:   AUAGACGAACGGCGCGUCUUUAAACCAUGCGUCGGGAGCGCGGCGGGUUC...
  ✗ Could not find CIF sequence in query for 8CH6_d
    CIF seq: GUGCUCGCUUCGGCAGCACAUAUACUAAAAUUGGAACGAUACAGAGAAGA...
    Query:   GUGCUCGCUUCGGCAGCACAUAUACUAAAAUUGGAACGAUACAGAGAAGA...
  ✗ Could not find CIF sequence in query for 7Y7C_V
    CIF seq: UCCUCUUGUAAGUGGGAGUAUCCCCGCCUGUCAGCGGGAGAGGGGCGUUC...
    Query:   UCCUCGUUAGUAUAGUGGUGAGUAUCCCCGCCUGUCACGCGGGAGACCGG...
  ✗ Could not find CIF sequence in query for 7XHT_B
    CIF seq: GCGGAUAACAAUUCCCCGGCUCUUCCAACUUAGGUUGAAAGAGCACAGGC...
    Query:   UGUGAGCGGAUAACAAUUCCCCGGCUCUUCCAACUUUAUGGUUGCGACCG...
  ✗ Could not 

In [32]:
import os
import glob
from Bio.PDB import MMCIFParser

# ——— CONFIGURATION ———
CIF_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/Test_dataset'
MSA_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA'
CROPPED_MSA_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/CROPPED_MSA'

# Create output directory if it doesn't exist
os.makedirs(CROPPED_MSA_DIR, exist_ok=True)

parser = MMCIFParser(QUIET=True)

def extract_rna_sequence(cif_path: str) -> str:
    """Extract only A/C/G/U from the first model of the CIF."""
    structure = parser.get_structure('', cif_path)
    seq = []
    for chain in structure[0]:
        for residue in chain:
            res = residue.get_resname().strip()
            if res in ('A','C','G','U'):
                seq.append(res)
    return ''.join(seq)

def read_msa_file(msa_path: str) -> list:
    """Read the entire MSA file and return list of (header, sequence) tuples."""
    sequences = []
    current_header = None
    current_seq = []
    
    with open(msa_path) as f:
        for line in f:
            line = line.strip()
            if line.startswith('>'):
                # Save previous sequence if exists
                if current_header is not None:
                    sequences.append((current_header, ''.join(current_seq)))
                # Start new sequence
                current_header = line
                current_seq = []
            else:
                current_seq.append(line)
        
        # Save last sequence
        if current_header is not None:
            sequences.append((current_header, ''.join(current_seq)))
    
    return sequences

def find_cif_sequence_in_query(cif_seq: str, query_seq: str) -> tuple:
    """Find the start and end positions of CIF sequence in query sequence."""
    if cif_seq in query_seq:
        start_pos = query_seq.find(cif_seq)
        end_pos = start_pos + len(cif_seq)
        return start_pos, end_pos
    return None, None

def crop_msa_alignment(msa_sequences: list, start_pos: int, end_pos: int) -> list:
    """Crop all sequences in the MSA to the specified positions."""
    cropped_sequences = []
    for header, sequence in msa_sequences:
        cropped_seq = sequence[start_pos:end_pos]
        cropped_sequences.append((header, cropped_seq))
    return cropped_sequences

def write_cropped_msa(cropped_sequences: list, output_path: str):
    """Write the cropped MSA to a new file."""
    with open(output_path, 'w') as f:
        for header, sequence in cropped_sequences:
            f.write(f"{header}\n")
            f.write(f"{sequence}\n")

total = 0
subset_count = 0
cropped_count = 0
no_msa_count = 0
failed_crop_count = 0
no_subset_ids = []

print("Processing CIF files and checking MSA compatibility...")
print("=" * 60)

for cif_file in glob.glob(os.path.join(CIF_DIR, '*.cif')):
    tid = os.path.splitext(os.path.basename(cif_file))[0]
    msa_file = os.path.join(MSA_DIR, f"{tid}.MSA.fasta")
    
    if not os.path.exists(msa_file):
        # skip entries with no MSA
        no_msa_count += 1
        continue

    cif_seq = extract_rna_sequence(cif_file)
    msa_sequences = read_msa_file(msa_file)
    
    if not msa_sequences:
        print(f"  Warning: Empty MSA file for {tid}")
        continue
    
    # Get the query sequence (first sequence)
    query_header, query_seq = msa_sequences[0]
    total += 1

    # Check if CIF sequence is a substring of query sequence
    if cif_seq in query_seq:
        subset_count += 1
    else:
        no_subset_ids.append(tid)
        
        # Try to find the CIF sequence in the query and crop accordingly
        start_pos, end_pos = find_cif_sequence_in_query(cif_seq, query_seq)
        
        if start_pos is not None:
            # Crop all sequences in the MSA
            cropped_sequences = crop_msa_alignment(msa_sequences, start_pos, end_pos)
            
            # Write cropped MSA
            output_path = os.path.join(CROPPED_MSA_DIR, f"{tid}.MSA.fasta")
            write_cropped_msa(cropped_sequences, output_path)
            cropped_count += 1
            print(f"  ✓ Cropped MSA for {tid}: positions {start_pos}-{end_pos}")
        else:
            # CIF sequence not found as substring - this shouldn't happen with your logic
            # but let's handle it gracefully
            print(f"  ✗ Could not find CIF sequence in query for {tid}")
            print(f"    CIF seq: {cif_seq[:50]}{'...' if len(cif_seq) > 50 else ''}")
            print(f"    Query:   {query_seq[:50]}{'...' if len(query_seq) > 50 else ''}")
            failed_crop_count += 1

# Final reporting
print("\n" + "=" * 60)
print("SUMMARY REPORT:")
print("=" * 60)
print(f"Total CIF files found:                  {total + no_msa_count}")
print(f"  - Files with MSA available:           {total}")
print(f"  - Files without MSA:                  {no_msa_count}")
print(f"\nMSA Compatibility Results:")
print(f"  ✓ CIF sequence found in MSA query:    {subset_count}")
print(f"  📝 Required cropping:                  {len(no_subset_ids)}")
print(f"    - Successfully cropped:             {cropped_count}")
print(f"    - Failed to crop:                   {failed_crop_count}")
print(f"\nOutput directory: {CROPPED_MSA_DIR}")

if cropped_count > 0:
    print(f"\nSuccessfully created {cropped_count} cropped MSA files")

if no_subset_ids:
    print(f"\nFiles that required cropping (first 10 of {len(no_subset_ids)}):")
    for tid in no_subset_ids[:10]:
        print(f"  - {tid}")
    
    if len(no_subset_ids) > 10:
        print(f"  ... and {len(no_subset_ids) - 10} more")

print("\n" + "=" * 60)
print("Process completed!")

Processing CIF files and checking MSA compatibility...
  ✗ Could not find CIF sequence in query for 9EY1_T
    CIF seq: UAAAUAUGUUUAACCAAAACAUCAGAUUGUGAAUCUGACAACAGAGGCUU...
    Query:   UAAAUAUAGUUUAACCAAAACAUCAGAUUGUGAAUCUGACAACAGAGGCU...
  ✗ Could not find CIF sequence in query for 9J6Y_E
    CIF seq: ACCGAUGAAGCUAGUGGAUAAGGUGUGACAAGCCGCCUAGCCAUACGUCU...
    Query:   UUAGUAUAUAAGUGUACCGAUGAAGCUAGUGGAUAAGGUGUGACAAGCCG...
  ✗ Could not find CIF sequence in query for 9LMF_F
    CIF seq: CGUUGCGCAUUUUGUUGCUCAAAAGGCGACGAAACGCAAGGCAAUGCACG...
    Query:   GCCGUCUCAAUAGUGGCUUAGCACAGAUAAUCCAUAGCGAUAUGGGAAAG...
  ✗ Could not find CIF sequence in query for 8X9M_A
    CIF seq: GCCGGGGCGCCACCCCGGAAGUGAUGCGAGUCGCAACUCGCAUCACAAGC...
    Query:   GCCGGGGCGCCACCCCGGAAGUGAUGCGAGUCGCAACUCGCAUCACAAGC...
  ✗ Could not find CIF sequence in query for 9L5R_2
    CIF seq: AGCUCUCUUUGCCUUUUGGCUUAGAUCAAGUGUAGUAUCUGUUUAAUCUC...
    Query:   AGCUCUCUUUGCCUUUUGGCUUAGAUCAAGUGUAGUAUCUGUUCUUUUCA...
  ✗ Could not f

# Generate MSA files for the missing

In [ ]:
import os
import glob
from Bio.PDB import MMCIFParser

# 1) Configuration
CIF_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/Train_dataset'
MSA_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA'

os.makedirs(MSA_DIR, exist_ok=True)
parser = MMCIFParser(QUIET=True)

def extract_rna_sequence(cif_path: str) -> str:
    """Pull out only the A/C/G/U residues in order from the first model."""
    structure = parser.get_structure('', cif_path)
    seq = []
    model = structure[0]
    for chain in model:
        for residue in chain:
            res = residue.get_resname().strip()
            if res in ('A', 'C', 'G', 'U'):
                seq.append(res)
    return ''.join(seq)

# 2) Iterate all .cif files
for cif_file in glob.glob(os.path.join(CIF_DIR, '*.cif')):
    tid = os.path.splitext(os.path.basename(cif_file))[0]
    msa_path = os.path.join(MSA_DIR, f"{tid}.MSA.fasta")

    # 3) If missing, write a single‐sequence MSA
    if not os.path.exists(msa_path):
        seq = extract_rna_sequence(cif_file)
        if not seq:
            print(f"⚠️  Skipping {tid}: no A/C/G/U residues found")
            continue

        with open(msa_path, 'w') as f:
            f.write(">query\n")
            f.write(seq + "\n")
        print(f"[+] Created missing MSA: {msa_path}")


In [4]:
import os
import pandas as pd

# 1) Configuration
MSA_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA'  # folder where .MSA.fasta files should live
SEQ_CSV = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/train_sequences.csv'

# 2) Load your sequence CSV
seq_df = pd.read_csv(SEQ_CSV, dtype=str)

# **Assumption**: your sequence column is named 'sequence'. 
# If your column is called something else, change the line below.
SEQ_COL = 'sequence'

# 3) Ensure MSA files exist for every target_id
for _, row in seq_df.iterrows():
    tid = row['target_id']
    msa_path = os.path.join(MSA_DIR, f"{tid}.MSA.fasta")
    if not os.path.exists(msa_path):
        seq = row[SEQ_COL]
        with open(msa_path, 'w') as f:
            f.write(f">query\n")
            f.write(f"{seq}\n")
        print(f"[+] Created missing MSA: {msa_path}")

[+] Created missing MSA: /home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA/8UU4_A.MSA.fasta
[+] Created missing MSA: /home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA/8JDJ_D.MSA.fasta
[+] Created missing MSA: /home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA/8AGZ_f.MSA.fasta
[+] Created missing MSA: /home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA/7NWW_1.MSA.fasta
[+] Created missing MSA: /home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA/9AX7_a.MSA.fasta
[+] Created missing MSA: /home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA/8G2B_1A.MSA.fasta
[+] Created missing MSA: /home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA/8PV3_C1.MSA.fasta
[+] Created missing MSA: /home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA/9BDL_A28S.MSA.fasta
[+] Created missing MSA: /home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA/7RQ9_2A.MSA.fasta
[+] Created missing MSA: /home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA/7QGG_t.MSA.

In [ ]:
import os
import pandas as pd

# 1) Configuration
MSA_DIR = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA'  # folder where .MSA.fasta files should live
SEQ_CSV = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/train_sequences.v2.csv'
LBL_CSV = r'/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/train_labels.v2.csv'

# make sure the MSA directory exists
os.makedirs(MSA_DIR, exist_ok=True)

# the prefixes (lowercased) that define your validation set
prefixes = ['8uo6', '9dcf', '9b0l', '9bzc', '9kpo_b', '9j6y','9b2K','8vxz','8zmh','8ztv_y','9imb']
# prefixes = ['8uo6', '9dcf', '9b0l', '9bzc', '9bz1', '9j6y', '9j3t','9b2K','8vxz','8zmh']
prefixes = [p.lower() for p in prefixes]

# 2) Load your sequence CSV
seq_df = pd.read_csv(SEQ_CSV, dtype=str)

# **Assumption**: your sequence column is named 'sequence'. 
# If your column is called something else, change the line below.
SEQ_COL = 'sequence'

# 3) Ensure MSA files exist for every target_id
for _, row in seq_df.iterrows():
    tid = row['target_id']
    msa_path = os.path.join(MSA_DIR, f"{tid}.MSA.fasta")
    if not os.path.exists(msa_path):
        seq = row[SEQ_COL]
        with open(msa_path, 'w') as f:
            f.write(f">query\n")
            f.write(f"{seq}\n")
        print(f"[+] Created missing MSA: {msa_path}")

# 4) Split sequences into train vs. validation
mask_seq = seq_df['target_id'].str.lower().str.startswith(tuple(prefixes))
validation_seq = seq_df[mask_seq]
train_seq      = seq_df[~mask_seq]

validation_seq.to_csv('validation_sequences.csv', index=False)
train_seq     .to_csv('train_sequences.csv',      index=False)

# 5) Load your labels CSV and split on the same prefixes
lab_df = pd.read_csv(LBL_CSV, dtype=str)
mask_lab = lab_df['ID'].str.lower().str.startswith(tuple(prefixes))

validation_lab = lab_df[mask_lab]
train_lab      = lab_df[~mask_lab]

validation_lab.to_csv('validation_labels.csv', index=False)
train_lab     .to_csv('train_labels.csv',    index=False)

print("Done! Generated:\n"
      "  – msa/*.MSA.fasta (added any missing)\n"
      "  – train_filtered.csv\n"
      "  – validation_filtered.csv\n"
      "  – train_lables_filtered.csv\n"
      "  – validation_labels_filtered.csv")


[+] Created missing MSA: /home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA/4WQ1_14.MSA.fasta
[+] Created missing MSA: /home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA/4V7H_B5.MSA.fasta
[+] Created missing MSA: /home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA/2O44_A.MSA.fasta
[+] Created missing MSA: /home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA/6MTB_5.MSA.fasta
[+] Created missing MSA: /home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA/4V9D_CA.MSA.fasta
[+] Created missing MSA: /home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA/8UU4_A.MSA.fasta
[+] Created missing MSA: /home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA/5J7L_DA.MSA.fasta
[+] Created missing MSA: /home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA/6HTQ_A.MSA.fasta
[+] Created missing MSA: /home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA/4V4Q_BB.MSA.fasta
[+] Created missing MSA: /home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/MSA/4LNT_YA.MSA.

In [12]:
import pandas as pd
df = pd.read_csv(r"/home/ubuntu/shafi_workspace/Protenix-RNA-Kaggle/data/validation_sequences.csv")

In [13]:
df.shape

(11, 5)

In [14]:
df['temporal_cutoff'] = pd.to_datetime(df['temporal_cutoff'], dayfirst=True)

# 3) Define the cutoff threshold
cutoff_date = pd.Timestamp('2024-09-18')

# 4) Filter rows where cutoff is before September 18, 2024
df = df[df['temporal_cutoff'] <= cutoff_date]

/tmp/ipykernel_1926/2743306250.py:1: UserWarning: Parsing dates in %Y-%m-%d format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df['temporal_cutoff'] = pd.to_datetime(df['temporal_cutoff'], dayfirst=True)
